# Read depth from a BAM, straight from pysam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/05_bam_coverage.ipynb)

[pysam](https://pysam.readthedocs.io) is how Python reads BAM and CRAM. `count_coverage` over a region, bin it, and `add_features` puts it on the genome — no intermediate file, no bigWig conversion. The data is the real 1000 Genomes **NA12878 exome** (a 17 GB BAM, but pysam fetches only the index and the region you ask for).

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy pysam

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Coverage over BRCA1

Open the remote BAM (its `.bai` is fetched automatically), sum the per-base A/C/G/T counts, and average into 100 bp bins. This BAM is aligned to GRCh37 and names the chromosome `17`; we relabel to `chr17` to match the hg19 hub below.

In [ ]:
import numpy as np
import pandas as pd
import pysam

BAM = (
    "https://s3.amazonaws.com/1000genomes/phase3/data/NA12878/"
    "exome_alignment/NA12878.mapped.ILLUMINA.bwa.CEU.exome.20121211.bam"
)
CHROM, START, END = "17", 41_196_312, 41_277_500  # BRCA1, GRCh37

bam = pysam.AlignmentFile(BAM)
depth = np.array(bam.count_coverage(CHROM, START, END)).sum(0)

binsize = 100
n = depth.size // binsize * binsize
binned = depth[:n].reshape(-1, binsize).mean(1).round(1)
starts = START + np.arange(binned.size) * binsize
coverage = pd.DataFrame(
    {"chrom": "chr17", "start": starts, "end": starts + binsize, "depth": binned}
)
coverage.head()

## See it on hg19, opened at the gene by name

`fetch_hub("hg19")` brings the genome and a gene-name search index, so `location="BRCA1"` just works. Exome capture concentrates reads on the exons — the depth track peaks there and drops between.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, fetch_hub

hg19 = fetch_hub("hg19")
view = LinearGenomeView(
    assembly=hg19["assemblies"][0],
    aggregate_text_search_adapters=hg19["aggregateTextSearchAdapters"],
    location="BRCA1",
)
view.add_features(
    coverage, name="NA12878 exome depth",
    color="jexl:get(feature,'depth') > 40 ? '#c62828' : get(feature,'depth') > 10 ? '#f9a825' : '#cfcfcf'",
)
view